# 🔍 C3-Pipeline: SQL Analytics with DuckDB

## Overview
This notebook loads the processed data into DuckDB and runs advanced analytical queries
to answer key business questions about consult turnaround, communication friction,
and financial impact.

### Queries Covered
1. **Consult Turnaround by Specialty** — Which specialties are slowest?
2. **Channel Effectiveness** — Pager vs. Secure Chat response comparison
3. **Friction Analysis** — Do more messages = longer delays?
4. **Weekend vs. Weekday** — Staffing gap analysis
5. **Excess Bed-Day Cost** — Financial impact of consult delays
6. **Specialist Leaderboard** — Individual specialist performance ranking

## Setup

In [ ]:
# Install dependencies (uncomment for Google Colab)
# !pip install duckdb pandas

import duckdb
import pandas as pd
import os

# Connect to DuckDB (in-memory for Colab, or file-based)
con = duckdb.connect('c3_pipeline.duckdb')

print(f'✅ DuckDB connected (version: {duckdb.__version__})')

## Load Processed Data into DuckDB

We load the CSV outputs from the PySpark ETL pipeline directly into DuckDB tables.

In [ ]:
CSV_DIR = os.path.join('data', 'processed', 'csv')

# Helper function to find the actual CSV file inside Spark's output directory
def find_csv_file(table_dir):
    """Spark writes CSVs as part-00000-*.csv inside the table directory."""
    for f in os.listdir(table_dir):
        if f.endswith('.csv') and f.startswith('part-'):
            return os.path.join(table_dir, f)
    # Fallback: try any .csv file
    for f in os.listdir(table_dir):
        if f.endswith('.csv'):
            return os.path.join(table_dir, f)
    raise FileNotFoundError(f'No CSV file found in {table_dir}')

# Tables to load
tables = [
    'dim_encounters',
    'fact_consult_orders',
    'fact_communication_logs',
    'fact_consult_completions',
    'agg_consult_friction',
    'agg_encounter_summary',
]

for table in tables:
    table_dir = os.path.join(CSV_DIR, table)
    csv_path = find_csv_file(table_dir)
    
    # Drop existing table and create from CSV
    con.execute(f'DROP TABLE IF EXISTS {table}')
    con.execute(f"CREATE TABLE {table} AS SELECT * FROM read_csv_auto('{csv_path}')")
    
    count = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'✅ {table}: {count:,} rows loaded')

print('\n🎉 All tables loaded into DuckDB!')

---
## Query 1: Consult Turnaround Time by Specialty

**Business Question**: What is the average consult turnaround time (order to note signed) by specialty?

**Why it matters**: Hospital SLAs typically target < 4 hours from consult order to specialist bedside arrival. Specialties consistently exceeding this threshold create patient care delays and extended hospital stays.

**Expected Finding**: Cardiology and Psychiatry should emerge as the slowest specialties.

In [ ]:
query_1 = """
SELECT
    co.target_specialty,
    COUNT(*) AS total_consults,
    ROUND(AVG(cc.time_to_bedside_hours), 2) AS avg_hours_to_bedside,
    ROUND(AVG(cc.time_to_note_signed_hours), 2) AS avg_hours_to_note_signed,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY cc.time_to_bedside_hours), 2) AS median_hours_to_bedside,
    ROUND(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY cc.time_to_bedside_hours), 2) AS p90_hours_to_bedside
FROM fact_consult_orders co
JOIN fact_consult_completions cc ON co.consult_order_id = cc.consult_order_id
WHERE co.order_status = 'Completed'
GROUP BY co.target_specialty
ORDER BY avg_hours_to_bedside DESC;
"""

df_q1 = con.execute(query_1).fetchdf()
print('📊 QUERY 1: Average Consult Turnaround Time by Specialty')
print('=' * 80)
print(df_q1.to_string(index=False))
print()

# Interpretation
slowest = df_q1.iloc[0]
fastest = df_q1.iloc[-1]
print(f'🔴 Slowest: {slowest["target_specialty"]} at {slowest["avg_hours_to_bedside"]:.1f} hours average')
print(f'🟢 Fastest: {fastest["target_specialty"]} at {fastest["avg_hours_to_bedside"]:.1f} hours average')
print(f'   Gap: {slowest["avg_hours_to_bedside"] - fastest["avg_hours_to_bedside"]:.1f} hours difference')

---
## Query 2: Communication Channel Effectiveness

**Business Question**: How does communication channel choice affect specialist response time?

**Why it matters**: Hospitals investing in modern secure messaging platforms need data to justify the ROI. If Secure App Chat demonstrably outperforms Legacy Pagers, this provides a strong case for full platform migration.

**Expected Finding**: Vocera Badge Call (real-time voice) should be fastest, followed by Secure App Chat, with Legacy Pager and Phone Call significantly lagging.

In [ ]:
query_2 = """
SELECT
    cl.channel,
    COUNT(*) AS total_messages,
    SUM(CASE WHEN cl.message_read_timestamp IS NOT NULL THEN 1 ELSE 0 END) AS messages_read,
    ROUND(100.0 * SUM(CASE WHEN cl.message_read_timestamp IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS read_rate_pct,
    ROUND(AVG(cl.response_lag_minutes), 1) AS avg_response_lag_min,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY cl.response_lag_minutes), 1) AS median_response_lag_min
FROM fact_communication_logs cl
WHERE cl.response_lag_minutes IS NOT NULL
GROUP BY cl.channel
ORDER BY avg_response_lag_min ASC;
"""

df_q2 = con.execute(query_2).fetchdf()
print('📊 QUERY 2: Channel Effectiveness — Response Lag & Read Rate')
print('=' * 80)
print(df_q2.to_string(index=False))
print()

# Interpretation
if len(df_q2) >= 2:
    best = df_q2.iloc[0]
    worst = df_q2.iloc[-1]
    reduction_pct = (1 - best['avg_response_lag_min'] / worst['avg_response_lag_min']) * 100
    print(f'💡 Insight: {best["channel"]} is {reduction_pct:.0f}% faster than {worst["channel"]}')
    print(f'   {best["channel"]}: {best["avg_response_lag_min"]:.1f} min avg response')
    print(f'   {worst["channel"]}: {worst["avg_response_lag_min"]:.1f} min avg response')

---
## Query 3: Friction Analysis

**Business Question**: Is there a correlation between the number of nurse outreach messages and the consult turnaround time?

**Why it matters**: If more messages correlate with longer delays, it validates that "telephone tag" is a symptom of an unresponsive system, not a workaround that helps. This justifies investment in notification escalation workflows.

**Expected Finding**: Consults requiring 6+ messages should have significantly longer time-to-bedside than those resolved with 1 message.

In [ ]:
query_3 = """
WITH friction_buckets AS (
    SELECT
        af.consult_order_id,
        af.friction_score,
        CASE
            WHEN af.friction_score = 1 THEN '1 message'
            WHEN af.friction_score BETWEEN 2 AND 3 THEN '2-3 messages'
            WHEN af.friction_score BETWEEN 4 AND 5 THEN '4-5 messages'
            ELSE '6+ messages'
        END AS friction_bucket,
        cc.time_to_bedside_hours
    FROM agg_consult_friction af
    JOIN fact_consult_completions cc ON af.consult_order_id = cc.consult_order_id
)
SELECT
    friction_bucket,
    COUNT(*) AS consult_count,
    ROUND(AVG(time_to_bedside_hours), 2) AS avg_hours_to_bedside,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY time_to_bedside_hours), 2) AS median_hours_to_bedside
FROM friction_buckets
GROUP BY friction_bucket
ORDER BY avg_hours_to_bedside ASC;
"""

df_q3 = con.execute(query_3).fetchdf()
print('📊 QUERY 3: Friction Analysis — Messages vs. Delay')
print('=' * 80)
print(df_q3.to_string(index=False))
print()

# Interpretation
if len(df_q3) >= 2:
    low_friction = df_q3[df_q3['friction_bucket'] == '1 message']
    high_friction = df_q3[df_q3['friction_bucket'] == '6+ messages']
    if len(low_friction) > 0 and len(high_friction) > 0:
        diff = high_friction.iloc[0]['avg_hours_to_bedside'] - low_friction.iloc[0]['avg_hours_to_bedside']
        print(f'💡 Insight: High-friction consults (6+ messages) take {diff:.1f} hours longer than low-friction (1 message)')

---
## Query 4: Weekend vs. Weekday Analysis

**Business Question**: How much slower are consults completed on weekends vs. weekdays?

**Why it matters**: Weekend staffing decisions have direct financial implications. If weekend consults are significantly delayed, it may justify additional on-call specialist coverage.

**Expected Finding**: Weekend + Night Shift should be the slowest combination.

In [ ]:
query_4 = """
SELECT
    CASE
        WHEN EXTRACT(DOW FROM co.order_timestamp) IN (0, 6) THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type,
    CASE
        WHEN EXTRACT(HOUR FROM co.order_timestamp) BETWEEN 7 AND 18 THEN 'Day Shift (7a-7p)'
        ELSE 'Night Shift (7p-7a)'
    END AS shift,
    COUNT(*) AS total_consults,
    ROUND(AVG(cc.time_to_bedside_hours), 2) AS avg_hours_to_bedside,
    ROUND(AVG(af.friction_score), 2) AS avg_messages_per_consult
FROM fact_consult_orders co
JOIN fact_consult_completions cc ON co.consult_order_id = cc.consult_order_id
JOIN agg_consult_friction af ON co.consult_order_id = af.consult_order_id
WHERE co.order_status = 'Completed'
GROUP BY day_type, shift
ORDER BY avg_hours_to_bedside DESC;
"""

df_q4 = con.execute(query_4).fetchdf()
print('📊 QUERY 4: Weekend vs. Weekday Consult Turnaround')
print('=' * 80)
print(df_q4.to_string(index=False))
print()

# Interpretation
if len(df_q4) >= 2:
    slowest_combo = df_q4.iloc[0]
    fastest_combo = df_q4.iloc[-1]
    print(f'💡 Insight: {slowest_combo["day_type"]} {slowest_combo["shift"]} is the slowest at {slowest_combo["avg_hours_to_bedside"]:.1f} hours')
    print(f'   vs. {fastest_combo["day_type"]} {fastest_combo["shift"]} at {fastest_combo["avg_hours_to_bedside"]:.1f} hours')

---
## Query 5: Excess Bed-Day Cost Estimation

**Business Question**: What is the estimated financial cost of consult-related discharge delays?

**Why it matters**: This translates abstract "consult delay" into concrete dollar figures that hospital COOs and CFOs can act on. At $2,500/bed-day (Florida average), even modest improvements can save millions annually.

**Assumption**: Average hospital bed cost = $2,500/day.

In [ ]:
query_5 = """
WITH encounter_delays AS (
    SELECT
        de.encounter_id,
        de.admitting_unit,
        de.primary_diagnosis_desc,
        aes.length_of_stay_days,
        aes.estimated_excess_bed_days,
        aes.total_nurse_messages_sent,
        ROUND(aes.estimated_excess_bed_days * 2500, 2) AS estimated_excess_cost_usd
    FROM dim_encounters de
    JOIN agg_encounter_summary aes ON de.encounter_id = aes.encounter_id
    WHERE de.discharge_timestamp IS NOT NULL
      AND aes.estimated_excess_bed_days > 0.5
)
SELECT
    admitting_unit,
    COUNT(*) AS encounters_with_delay,
    ROUND(AVG(estimated_excess_bed_days), 2) AS avg_excess_bed_days,
    ROUND(SUM(estimated_excess_cost_usd), 0) AS total_estimated_cost_usd,
    ROUND(AVG(total_nurse_messages_sent), 1) AS avg_nurse_messages
FROM encounter_delays
GROUP BY admitting_unit
ORDER BY total_estimated_cost_usd DESC;
"""

df_q5 = con.execute(query_5).fetchdf()
print('📊 QUERY 5: Excess Bed-Day Cost Estimation by Admitting Unit')
print('=' * 80)
print(df_q5.to_string(index=False))
print()

# Interpretation
total_cost = df_q5['total_estimated_cost_usd'].sum()
print(f'💰 Total Estimated Excess Bed Cost: ${total_cost:,.0f}')
print(f'   Annualized (projected): ${total_cost:,.0f}')
print(f'   This represents the financial opportunity for improvement.')

---
## Query 6: Specialist Performance Leaderboard

**Business Question**: Which individual specialists have the longest average response times?

**Why it matters**: Individual accountability drives behavior change. By ranking specialists within their departments, hospital leadership can identify outliers who may need coaching, additional support, or schedule adjustments.

In [ ]:
query_6 = """
WITH specialist_stats AS (
    SELECT
        cc.specialist_id,
        cc.specialist_name,
        co.target_specialty,
        COUNT(*) AS consults_completed,
        ROUND(AVG(cc.time_to_bedside_hours), 2) AS avg_hours_to_bedside,
        ROUND(AVG(af.friction_score), 2) AS avg_friction_score
    FROM fact_consult_completions cc
    JOIN fact_consult_orders co ON cc.consult_order_id = co.consult_order_id
    JOIN agg_consult_friction af ON co.consult_order_id = af.consult_order_id
    GROUP BY cc.specialist_id, cc.specialist_name, co.target_specialty
    HAVING COUNT(*) >= 10
)
SELECT
    *,
    RANK() OVER (PARTITION BY target_specialty ORDER BY avg_hours_to_bedside DESC) AS rank_in_specialty
FROM specialist_stats
ORDER BY target_specialty, rank_in_specialty;
"""

df_q6 = con.execute(query_6).fetchdf()
print('📊 QUERY 6: Specialist Performance Leaderboard')
print('=' * 80)
print(f'Total specialists with >= 10 completed consults: {len(df_q6)}')
print()

# Show top 3 slowest per specialty
for specialty in df_q6['target_specialty'].unique():
    spec_data = df_q6[df_q6['target_specialty'] == specialty].head(3)
    print(f'\n  {specialty} (Top 3 Slowest):')
    for _, row in spec_data.iterrows():
        print(f'    #{int(row["rank_in_specialty"])}: {row["specialist_name"]} — '
              f'{row["avg_hours_to_bedside"]:.1f} hrs avg, '
              f'{int(row["consults_completed"])} consults, '
              f'friction {row["avg_friction_score"]:.1f}')

---
## Executive Summary

In [ ]:
print('=' * 60)
print('📋 C3-PIPELINE: EXECUTIVE SUMMARY')
print('=' * 60)

# KPI 1: Avg Time to Bedside
avg_bedside = con.execute(
    "SELECT ROUND(AVG(time_to_bedside_hours), 2) FROM fact_consult_completions"
).fetchone()[0]
print(f'\n⏱️  Avg. Time to Bedside: {avg_bedside:.1f} hours')
if avg_bedside > 6:
    print('   🔴 CRITICAL — Exceeds 6-hour threshold')
elif avg_bedside > 4:
    print('   🟡 WARNING — Above 4-hour SLA target')
else:
    print('   🟢 ON TARGET — Below 4-hour SLA')

# KPI 2: Avg Messages per Consult
avg_msgs = con.execute(
    "SELECT ROUND(AVG(friction_score), 2) FROM agg_consult_friction"
).fetchone()[0]
print(f'\n📨 Avg. Nurse Messages per Consult: {avg_msgs:.1f}')
if avg_msgs > 4:
    print('   🔴 HIGH FRICTION — Excessive communication burden')
elif avg_msgs > 2:
    print('   🟡 MODERATE — Room for improvement')
else:
    print('   🟢 LOW FRICTION')

# KPI 3: Message Read Rate
read_rate = con.execute("""
    SELECT ROUND(100.0 * SUM(CASE WHEN message_read_timestamp IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1)
    FROM fact_communication_logs
""").fetchone()[0]
print(f'\n📖 Message Read Rate: {read_rate:.1f}%')
if read_rate < 80:
    print('   🔴 CRITICAL — Too many unread messages')
elif read_rate < 90:
    print('   🟡 WARNING — Below 90% target')
else:
    print('   🟢 GOOD — Above 90% target')

# KPI 4: Total Excess Bed Cost
total_cost = con.execute("""
    SELECT ROUND(SUM(estimated_excess_bed_days * 2500), 0)
    FROM agg_encounter_summary
    WHERE estimated_excess_bed_days > 0.5
""").fetchone()[0]
print(f'\n💰 Estimated Excess Bed Cost: ${total_cost:,.0f}')
print('   🔴 This represents avoidable cost from consult-related delays')

print('\n' + '=' * 60)

In [ ]:
# Close connection
con.close()
print('✅ DuckDB connection closed.')